# Muhr — E10: maqola rasmi uchun belgilangan tasvir namunalari

Bu tajriba statistika uchun emas. T-COCO'dan 6 ta tasvir olinadi va har biri uchun quyidagilar saqlanadi: asl tasvir, bizning tarmoq + G9, Pixel Seal + G9, Video Seal + G9. Har bir variantning PSNR/SSIM/LPIPS qiymati va toza belgilangan tasvir kalit bilan tekshiruvdan o'tgani ham yoziladi.

**Input:** `muhr-v2` (**2.7** versiya), `best.pt` manbasi, COCO 2017. **GPU T4, Internet: On.** Taxminiy vaqt ≈ 10 daqiqa [taxmin].
Natija: `e10_examples.zip`. Shuni menga yuboring.


In [ ]:
# ===== Sozlamalar =====
STAGE = "e10_examples"
INDICES = "0,7,23,41,58,96"
VS_COMMIT = "f12875a9b5e711e0c6e2dc3a22aca18231ce4a1a"           # Video Seal kodi (bl_* bilan bir xil)
HF_REPO, HF_REV = "tangtianzhong/img-wm-torchscript", "845dc751783db2a03a4b14ea600b0a4a9aba89aa"


In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path
W, IN, TMP = Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/tmp")
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"buyruq xato bilan tugadi: {cmd[:3]}")

def find_dir(name):
    hits = [h for h in sorted(glob.glob(str(IN / "**" / name), recursive=True)) if os.path.isdir(h)]
    return Path(hits[0]) if hits else None

def _ver(d):
    try:
        return tuple(int(x) for x in (d / "VERSION").read_text().strip().split("."))
    except Exception:
        return (0,)
src = sorted({Path(p).parents[1] for p in glob.glob(str(IN / "**/muhr_proj/muhr_exp/codec2.py"), recursive=True)},
             key=_ver, reverse=True)
assert src, "muhr-v2 dataseti (muhr_proj/muhr_exp) topilmadi — Add Input qiling"
print("kod:", src[0], "versiya", ".".join(map(str, _ver(src[0]))))
assert _ver(src[0]) >= (2, 7), "muhr-v2 eski versiyada — Input'da 2.7 versiyani tanlang"
CODE = W / "muhr_proj"
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(src[0], CODE)
ck = sorted(glob.glob(str(IN / "**/runs/taklif/best.pt"), recursive=True))
assert ck, "runs/taklif/best.pt topilmadi"
CKPT = ck[0]; print("checkpoint:", CKPT)
run([sys.executable, "-m", "pip", "install", "-q", "galois", "lpips", "bchlib", "omegaconf", "einops", "pytorch_msssim",
     "pycocotools", "av", "decord"])


In [ ]:
# --- Video Seal kodi va baseline og'irliklari
VS = TMP / "videoseal"
if not VS.exists():
    run(["git", "clone", "-q", "https://github.com/facebookresearch/videoseal", VS])
    run(["git", "checkout", "-q", VS_COMMIT], cwd=VS)
(VS / "ckpts").mkdir(exist_ok=True)
if not (VS / "ckpts" / "hidden_encoder_48b.pt").exists():
    from huggingface_hub import snapshot_download
    try:
        d = snapshot_download(HF_REPO, revision=HF_REV, local_dir=str(TMP / "hf_ts"))
    except Exception as e:
        print("pinned revision failed:", e, "-> main"); d = snapshot_download(HF_REPO, local_dir=str(TMP / "hf_ts"))
    for f in glob.glob(str(Path(d) / "**" / "*.pt"), recursive=True):
        shutil.copy(f, VS / "ckpts" / Path(f).name)
print(sorted(os.listdir(VS / "ckpts")))


In [ ]:
# --- ma'lumotlar: T-COCO (val2017[1500:3500]) ning birinchi N_IMAGES tasviri
def link_subset(files, dst):
    dst = Path(dst)
    if dst.exists():
        return dst
    dst.mkdir(parents=True)
    for f in files:
        os.symlink(f, dst / Path(f).name)
    return dst
val = find_dir("val2017"); assert val is not None, "COCO val2017 topilmadi"
vfiles = sorted(glob.glob(str(val / "*.jpg")))
T_COCO = link_subset(vfiles[1500:3500], TMP / "test_coco")

OUT = W / STAGE
OUT.mkdir(exist_ok=True)


In [ ]:
os.chdir(CODE)
run([sys.executable, "scripts/examples.py", "--ckpt", CKPT, "--vs-root", VS, "--data", T_COCO, "--indices", INDICES,
     "--out", OUT])


In [ ]:
os.chdir(W)
z = shutil.make_archive(str(W / STAGE), "zip", OUT)
print("saqlandi:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
print(open(OUT / "examples.json").read()[:3000])
